# Montra Hospitality Intel: NLP Text Preprocessing & Review Corpus
**Guest Experience Analytics & Topic Modelling**

This notebook prepares the unstructured guest review corpus for NLP and topic modelling. This process inspects corpus quality, removes duplicates, eliminates empty/whitespace reviews, and implements standard cleaning (removing HTML tags, URLs, emojis, and normalizing text).

In [2]:
import re
import os
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

# Load consolidated dataset using relative paths
input_path = "../data/processed/merged_raw.csv"
if not os.path.exists(input_path):
    input_path = "data/processed/merged_raw.csv"

merged = pd.read_csv(input_path)
print("Loaded dataset shape:", merged.shape)
merged.head()


Loaded dataset shape: (35000, 22)


In [3]:
# Inspect column names
print("Columns in dataset:")
print(list(merged.columns))


Columns in dataset:
['review_id', 'booking_id', 'guest_id', 'hotel_id', 'review_text', 'rating_score', 'review_date', 'sentiment_label', 'room_type', 'booking_channel', 'check_in_date', 'check_out_date', 'total_amount', 'stay_duration', 'customer_type', 'loyalty_status', 'country', 'previous_stays', 'hotel_name', 'location', 'star_rating', 'total_rooms']


### 1. Corpus Quality & Characteristic Audit
Check for missing review text, duplicate reviews, and evaluate the text length distribution.

In [5]:
# Inspect missing review text and duplicates
print("Missing review_text count:", merged['review_text'].isna().sum())
print("Duplicate reviews count:", merged['review_text'].duplicated().sum())


Missing review_text count: 0
Duplicate reviews count: 464


In [6]:
# Deduplicate reviews on review_text
review_corpus = merged.drop_duplicates(subset=['review_text']).reset_index(drop=True)
print("Unique review corpus shape:", review_corpus.shape)


Unique review corpus shape: (34536, 22)


In [7]:
# Remove records that contain only whitespace
review_corpus = review_corpus[
    review_corpus['review_text'].str.strip().ne("")
].copy()
print("Corpus after removing whitespace reviews:", review_corpus.shape)


Corpus after removing whitespace reviews: (34536, 22)


In [8]:
# Calculate character length of reviews
review_corpus['text_length'] = review_corpus['review_text'].str.len()
print("Review length summary statistics:")
display(review_corpus['text_length'].describe())


Review length summary statistics:
count    34536.000000
mean       172.817437
std         39.220502
min         67.000000
25%        144.000000
50%        172.000000
75%        204.000000
max        280.000000
Name: text_length, dtype: float64


### 2. Text Cleaning Pipeline
Standardized text cleaning pipeline:
1. Converts input to string.
2. Removes HTML tags (`<.*?>`).
3. Removes URLs (`https?://\S+|www\.\S+`).
4. Strips emojis by normalizing to ASCII.
5. Converts text to lowercase.
6. Retains alphanumeric characters and spaces (`[^a-z0-9\s]`).
7. Normalizes and trims whitespace.

In [10]:
def clean_review(text):
    # Convert to string
    text = str(text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    # Remove emojis and non-ascii characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    # Convert to lowercase
    text = text.lower()
    # Keep letters, numbers, and spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


### 3. Apply Text Preprocessing
Apply `clean_review` across the review text to generate `cleaned_review_text`.

In [12]:
# Apply cleaning function to review text
merged['cleaned_review_text'] = merged['review_text'].map(clean_review)

# Preview original vs cleaned review text
display(merged[['review_text', 'cleaned_review_text']].head(10))


                                         review_text                                cleaned_review_text
0  The overall price was fair for what we paid. M...  the overall price was fair for what we paid mi...
1  The room was nothing out of the ordinary for w...  the room was nothing out of the ordinary for w...
2  Unfortunately, the pool was outdated — it clea...  unfortunately the pool was outdated it clearly...
3  We were disappointed that the neighborhood was...  we were disappointed that the neighborhood was...
4  The room service was standard hotel fare for w...  the room service was standard hotel fare for w...
5  Getting around from the hotel was in a rather ...  getting around from the hotel was in a rather ...
6  We couldn't have asked for a better base while...  we couldn t have asked for a better base while...
7  We had a wonderful time during our stay in Not...  we had a wonderful time during our stay in not...
8  The in-house cafe was acceptable for what we p...  the in hou

### 4. Save Cleaned NLP Dataset
Save the prepared NLP dataset to `../data/processed/montra_cleaned_data.csv`.

In [14]:
# Save cleaned dataset
os.makedirs("../data/processed", exist_ok=True)
output_path = "../data/processed/montra_cleaned_data.csv"
merged.to_csv(output_path, index=False)
print(f"Successfully saved {merged.shape[0]} cleaned records to {output_path}")


Successfully saved 35000 cleaned records to ../data/processed/montra_cleaned_data.csv
